# Media Mix Modelling (MMM)Analyze marketing channel effectiveness and optimize budget allocation using regression analysis.

In [ ]:
# Import librariesimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.linear_model import LinearRegression, Ridgefrom sklearn.preprocessing import StandardScalerimport statsmodels.api as smplt.style.use("seaborn-v0_8-whitegrid")

## 1. Load Data

In [ ]:
df = pd.read_csv('data/marketing_data.csv')print('Shape:', df.shape)print('\nColumns:', df.columns.tolist())df.head()

## 2. Exploratory Analysis

In [ ]:
# Check for duplicates and missing valuesprint('Missing values:')print(df.isnull().sum())print('\nBasic Statistics:')df.describe()

## 3. Correlation Analysis

In [ ]:
# Correlation between channels and saleschannels = ['tv_spend', 'radio_spend', 'social_spend', 'search_spend', 'display_spend', 'email_spend']corr = df[channels + ['sales']].corr()plt.figure(figsize=(10, 8))sns.heatmap(corr, annot=True, cmap='coolwarm', center=0)plt.title('Correlation Matrix: Marketing Spend vs Sales')plt.tight_layout()plt.show()print('\nCorrelation with Sales:')print(corr['sales'].sort_values(ascending=False))

## 4. Build MMM Model

In [ ]:
# Prepare features and targetX = df[channels]y = df['sales']# Add constant for statsmodelsX_const = sm.add_constant(X)# Fit OLS modelmodel = sm.OLS(y, X_const).fit()print(model.summary())

## 5. Model Interpretation

In [ ]:
# Extract coefficients (ROAS - Return on Ad Spend)coefs = model.params[channels]roas = coefs / 1  # For every 1€ spent, get coef€ in salesprint('ROAS (Return on Ad Spend) by Channel:')for ch, r in roas.sort_values(ascending=False).items():    print(f'  {ch.replace("_spend", ""):12s}: {r:.2f}x')

## 6. Channel Contribution

In [ ]:
# Calculate contribution to total salescontribution = (coefs * df[channels].mean()) / (coefs * df[channels].mean()).sum() * 100print('\nContribution to Sales (%):')for ch, c in contribution.sort_values(ascending=False).items():    print(f'  {ch.replace("_spend", ""):12s}: {c:.1f}%')# Visualizeplt.figure(figsize=(10, 6))contribution.sort_values().plot(kind='barh', color='#3498db')plt.title('Marketing Channel Contribution to Sales')plt.xlabel('Contribution (%)')plt.tight_layout()plt.show()

## 7. Budget Optimization

In [ ]:
# Current budget allocationcurrent_budget = df[channels].mean().sum()current_allocation = df[channels].mean()print(f'Current Total Budget: €{current_budget:,.0f}')print('\nCurrent Allocation:')for ch, val in current_allocation.items():    pct = val / current_budget * 100    print(f'  {ch.replace("_spend", ""):12s}: €{val:,.0f} ({pct:.1f}%)')# Optimal allocation based on ROASroas_sorted = roas.sort_values(ascending=False)print('\nRecommended Priority (by ROAS):')for ch in roas_sorted.index:    print(f'  {ch.replace("_spend", ""):12s}: {roas[ch]:.2f}x ROI')

## 8. Visualization: Spend vs Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Spend distributionaxes[0].pie(current_allocation, labels=[c.replace('_spend', '') for c in channels],             autopct='%1.1f%%', colors=plt.cm.Set3.colors)axes[0].set_title('Current Budget Allocation')# ROI by channelroas.sort_values().plot(kind='barh', ax=axes[1], color='#2ecc71')axes[1].set_title('ROAS by Channel')axes[1].set_xlabel('Return per €1 Spent')plt.tight_layout()plt.show()

## 9. Key Findings & Recommendations

In [ ]:
print("""## 🔑 Key Findings### Most Effective Channels (ROAS):1. TV: Highest reach and brand awareness2. Search: High intent, direct response3. Social: Good for engagement### Least Effective Channels:- Display ads have lowest ROAS- Email has moderate impact### Model Performance:- R²: {:.3f}- The model explains {:.1f}% of sales variance## 💡 Recommendations### 1. Increase Budget:- TV and Search are top performers- Consider scaling successful campaigns### 2. Decrease Budget:- Display advertising underperforms- Test different creative or reduce spend### 3. Next Steps:- A/B test channel combinations- Add external factors (seasonality, holidays)- Build media mix optimization model""".format(model.rsquared, model.rsquared * 100))